In [1]:
import pandas as pd
# read csv file

df = pd.read_csv("../data/youtube_india_clean.csv")


print("Dataset shape:", df.shape)
# printing number of rows and columns so we know dataset loaded correctly


print(df.columns)
# checking which features are available

df.head()

Dataset shape: (302786, 15)
Index(['video_title', 'video_category_id', 'video_duration',
       'video_definition', 'video_view_count', 'video_like_count',
       'video_comment_count', 'video_published_at', 'video_trending__date',
       'channel_title', 'channel_view_count', 'channel_subscriber_count',
       'channel_video_count', 'channel_have_hidden_subscribers', 'video_tags'],
      dtype='object')


,video_title,video_category_id,video_duration,video_definition,video_view_count,video_like_count,video_comment_count,video_published_at,video_trending__date,channel_title,channel_view_count,channel_subscriber_count,channel_video_count,channel_have_hidden_subscribers,video_tags
0,Bhool Bhulaiyaa 3 (Official Trailer): Kartik A...,Music,PT3M51S,hd,56032799.0,1058450.0,44767.0,2024-10-09T09:00:08Z,2024.10.12,T-Series,2.693735e+11,276000000.0,21864.0,False,"tseries,tseries songs,bhool bhulaiyaa 3,bhool ..."
1,Singham Again | Official Trailer | A Rohit She...,Entertainment,PT4M58S,hd,40832089.0,951140.0,83563.0,2024-10-07T07:43:56Z,2024.10.12,JioStudios,4.907153e+08,794000.0,623.0,False,"singham again ajay devgn,singam 3 movie,singha..."
2,Bhool Bhulaiyaa 3 (Official Trailer): Kartik A...,Music,PT3M51S,hd,56032799.0,1058450.0,44767.0,2024-10-09T09:00:08Z,2024.10.12,T-Series,2.693735e+11,276000000.0,21864.0,False,"tseries,tseries songs,bhool bhulaiyaa 3,bhool ..."
3,Singham Again | Official Trailer | A Rohit She...,Entertainment,PT4M58S,hd,40832089.0,951140.0,83563.0,2024-10-07T07:43:56Z,2024.10.12,JioStudios,4.907153e+08,794000.0,623.0,False,"singham again ajay devgn,singam 3 movie,singha..."
4,Bhool Bhulaiyaa 3 (Official Trailer): Kartik A...,Music,PT3M51S,hd,56032799.0,1058450.0,44767.0,2024-10-09T09:00:08Z,2024.10.12,T-Series,2.693735e+11,276000000.0,21864.0,False,"tseries,tseries songs,bhool bhulaiyaa 3,bhool ..."


In [2]:
import re
# importing re because we need it to convert youtube duration text like PT3M51S into seconds

df_model2 = df.copy()
# making a copy so the original dataframe stays safe

# filling only the values we may use in this model
df_model2["video_view_count"] = df_model2["video_view_count"].fillna(df_model2["video_view_count"].median())
df_model2["video_category_id"] = df_model2["video_category_id"].fillna(df_model2["video_category_id"].mode()[0])
# filling missing target/category values so the model does not break later

# keeping only pre-upload useful columns + target
df_model2 = df_model2[[
    "video_category_id",
    "video_duration",
    "video_definition",
    "video_view_count",
    "channel_subscriber_count",
    "channel_video_count",
    "channel_have_hidden_subscribers"
]]
# keeping only columns that are known before upload
# not taking likes/comments/views-related post-upload signals as features

# function to convert PT3M51S style duration into total seconds
def convert_duration_to_seconds(duration):
    hours = 0
    minutes = 0
    seconds = 0

    h = re.search(r"(\d+)H", str(duration))
    m = re.search(r"(\d+)M", str(duration))
    s = re.search(r"(\d+)S", str(duration))

    if h:
        hours = int(h.group(1))
    if m:
        minutes = int(m.group(1))
    if s:
        seconds = int(s.group(1))

    return hours * 3600 + minutes * 60 + seconds

df_model2["video_duration_seconds"] = df_model2["video_duration"].apply(convert_duration_to_seconds)
# converting duration text into a numeric column the model can understand

df_model2 = df_model2.drop(columns=["video_duration"])
# dropping old text duration because we now have duration in seconds

print("Model 2 dataset shape:", df_model2.shape)
print("\nColumns in model 2 dataset:")
print(df_model2.columns)

df_model2.head()

Model 2 dataset shape: (302786, 7)

Columns in model 2 dataset:
Index(['video_category_id', 'video_definition', 'video_view_count',
       'channel_subscriber_count', 'channel_video_count',
       'channel_have_hidden_subscribers', 'video_duration_seconds'],
      dtype='object')


,video_category_id,video_definition,video_view_count,channel_subscriber_count,channel_video_count,channel_have_hidden_subscribers,video_duration_seconds
0,Music,hd,56032799.0,276000000.0,21864.0,False,231
1,Entertainment,hd,40832089.0,794000.0,623.0,False,298
2,Music,hd,56032799.0,276000000.0,21864.0,False,231
3,Entertainment,hd,40832089.0,794000.0,623.0,False,298
4,Music,hd,56032799.0,276000000.0,21864.0,False,231


In [3]:
# creating classification target (same idea as before)
median_views = df_model2["video_view_count"].median()
# finding middle value of views to split high vs low performance

df_model2["high_performance"] = (df_model2["video_view_count"] > median_views).astype(int)
# if views > median → 1 (high), else → 0 (low)

# encoding categorical columns
df_encoded2 = pd.get_dummies(
    df_model2,
    columns=["video_category_id", "video_definition"],
    drop_first=True
)
# converting text columns into numeric columns (0/1)
# model cannot understand text like "Music" or "hd"

print("Encoded dataset shape:", df_encoded2.shape)
print("\nColumns after encoding:")
print(df_encoded2.columns)

df_encoded2.head()

Encoded dataset shape: (302786, 20)

Columns after encoding:
Index(['video_view_count', 'channel_subscriber_count', 'channel_video_count',
       'channel_have_hidden_subscribers', 'video_duration_seconds',
       'high_performance', 'video_category_id_Comedy',
       'video_category_id_Education', 'video_category_id_Entertainment',
       'video_category_id_Film & Animation', 'video_category_id_Gaming',
       'video_category_id_Howto & Style', 'video_category_id_Music',
       'video_category_id_News & Politics', 'video_category_id_People & Blogs',
       'video_category_id_Pets & Animals',
       'video_category_id_Science & Technology', 'video_category_id_Sports',
       'video_category_id_Travel & Events', 'video_definition_sd'],
      dtype='object')


,video_view_count,channel_subscriber_count,channel_video_count,channel_have_hidden_subscribers,video_duration_seconds,high_performance,video_category_id_Comedy,video_category_id_Education,video_category_id_Entertainment,video_category_id_Film & Animation,video_category_id_Gaming,video_category_id_Howto & Style,video_category_id_Music,video_category_id_News & Politics,video_category_id_People & Blogs,video_category_id_Pets & Animals,video_category_id_Science & Technology,video_category_id_Sports,video_category_id_Travel & Events,video_definition_sd
0,56032799.0,276000000.0,21864.0,False,231,1,False,False,False,False,False,False,True,False,False,False,False,False,False,False
1,40832089.0,794000.0,623.0,False,298,1,False,False,True,False,False,False,False,False,False,False,False,False,False,False
2,56032799.0,276000000.0,21864.0,False,231,1,False,False,False,False,False,False,True,False,False,False,False,False,False,False
3,40832089.0,794000.0,623.0,False,298,1,False,False,True,False,False,False,False,False,False,False,False,False,False,False
4,56032799.0,276000000.0,21864.0,False,231,1,False,False,False,False,False,False,True,False,False,False,False,False,False,False


In [4]:
from sklearn.model_selection import train_test_split
# importing function to split data into train and test sets

# target (what we want to predict)
y2 = df_encoded2["high_performance"]
# this is the output label (0 or 1)

# features (input to model)
X2 = df_encoded2.drop(columns=["high_performance", "video_view_count"])
# removing:
# high_performance → because it's the target
# video_view_count → because target was created from it (avoid leakage)

# splitting data
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2,
    test_size=0.2,
    random_state=42
)
# 80% training, 20% testing

print("Train shape:", X2_train.shape)
print("Test shape:", X2_test.shape)

Train shape: (242228, 18)
Test shape: (60558, 18)


In [5]:
from sklearn.ensemble import RandomForestClassifier
# importing Random Forest model for classification

model2 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
# creating the model
# n_estimators=100 → number of trees
# n_jobs=-1 → use all CPU cores for faster training

model2.fit(X2_train, y2_train)
# training the model on training data

print("Model 2 trained successfully!")

Model 2 trained successfully!


In [6]:
from sklearn.metrics import accuracy_score, classification_report

# predictions
y2_pred = model2.predict(X2_test)
# predicting on test data

# accuracy
accuracy2 = accuracy_score(y2_test, y2_pred)
print("Accuracy:", accuracy2)

# detailed report
print("\nClassification Report:")
print(classification_report(y2_test, y2_pred))

Accuracy: 0.9789458040225899

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     30187
           1       0.98      0.98      0.98     30371

    accuracy                           0.98     60558
   macro avg       0.98      0.98      0.98     60558
weighted avg       0.98      0.98      0.98     60558



“big channel = high performance”

### Model 2 - (No Channel Bias)

In [8]:
# creating final cleaner version from model 2 data
df_final2 = df_model2.copy()
# copying current pre-upload dataset so old version stays safe

# dropping strong channel-size proxy column
df_final2 = df_final2.drop(columns=["channel_subscriber_count"])
# removing subscriber count because it is dominating the prediction
# we want the model to learn more from content/setup, not just channel size

print("Final model dataset shape:", df_final2.shape)
print("\nFinal model columns:")
print(df_final2.columns)

df_final2.head()

Final model dataset shape: (302786, 7)

Final model columns:
Index(['video_category_id', 'video_definition', 'video_view_count',
       'channel_video_count', 'channel_have_hidden_subscribers',
       'video_duration_seconds', 'high_performance'],
      dtype='object')


,video_category_id,video_definition,video_view_count,channel_video_count,channel_have_hidden_subscribers,video_duration_seconds,high_performance
0,Music,hd,56032799.0,21864.0,False,231,1
1,Entertainment,hd,40832089.0,623.0,False,298,1
2,Music,hd,56032799.0,21864.0,False,231,1
3,Entertainment,hd,40832089.0,623.0,False,298,1
4,Music,hd,56032799.0,21864.0,False,231,1


In [9]:
# encoding categorical columns again for final dataset
df_final_encoded = pd.get_dummies(
    df_final2,
    columns=["video_category_id", "video_definition"],
    drop_first=True
)
# converting category and definition into numeric columns (0/1)

# target
y_final = df_final_encoded["high_performance"]
# what we want to predict

# features
X_final = df_final_encoded.drop(columns=["high_performance", "video_view_count"])
# removing target and video_view_count (to avoid leakage)

from sklearn.model_selection import train_test_split

# split data
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_final, y_final,
    test_size=0.2,
    random_state=42
)

print("Train shape:", Xf_train.shape)
print("Test shape:", Xf_test.shape)

Train shape: (242228, 17)
Test shape: (60558, 17)


### Why Random Forest?

- **Mixed feature types** — features include categorical (category, definition) and numeric (duration, channel activity). Random Forest makes no assumption about linear relationships between features and the outcome, unlike Logistic Regression which would struggle here.
- **Avoids overfitting** — a single Decision Tree memorizes training data. Random Forest averages across 100 trees each trained on a random subset, which generalizes much better.
- **Parallel training** — all trees are trained independently (`n_jobs=-1` uses all CPU cores), making it fast on 300K rows. Sequential methods like Gradient Boosting would take significantly longer for a modest accuracy gain.
- **Built-in feature importance** — tells us which features actually drive the prediction, which is valuable for understanding what influences video performance.
- **Robust to outliers** — some videos have extreme durations; RF handles these without needing feature scaling unlike SVM or KNN.

In [10]:
from sklearn.ensemble import RandomForestClassifier
# importing Random Forest for classification

model_final = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
# creating the model
# same as before but now using cleaner features

model_final.fit(Xf_train, yf_train)
# training model on final dataset

print("Final model trained!")

Final model trained!


In [11]:
from sklearn.metrics import accuracy_score, classification_report

# predictions
yf_pred = model_final.predict(Xf_test)

# accuracy
accuracy_final = accuracy_score(yf_test, yf_pred)
print("Final Model Accuracy:", accuracy_final)

# detailed report
print("\nClassification Report:")
print(classification_report(yf_test, yf_pred))

Final Model Accuracy: 0.9687076851943591

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.96      0.97     30187
           1       0.96      0.98      0.97     30371

    accuracy                           0.97     60558
   macro avg       0.97      0.97      0.97     60558
weighted avg       0.97      0.97      0.97     60558



Accuracy ≈ 96.8%
No likes/comments/subscribers used
Model still performs very well

In [12]:
import joblib

joblib.dump(model_final, "../models/preupload_performance_model.pkl")

print("Final model saved!")

Final model saved!
